# Audit unifié A1-A7 — France (pipeline unique, hors-ligne, exact)

Ce notebook applique **une seule** fonction d'audit, `gbfs_toolkit.audit_static`, au catalogue français publié. C'est la même fonction que le notebook *Global* applique au reste du monde, ce qui rend les deux colonnes enfin comparables.

Source: `catalogue/stations_gold_standard_final.parquet` (station-level).

In [1]:
import pandas as pd
import unified_audit as ua
PARQUET = '../catalogue/stations_gold_standard_final.parquet'
print('Pipeline:', 'gbfs_toolkit.audit_static',
      '| a7_scope=', ua.A7_SCOPE, '| a4_sigma=', ua.A4_SIGMA)

Pipeline: gbfs_toolkit.audit_static | a7_scope= all | a4_sigma= 3.0


## 1. Audit (la fonction unique)

In [2]:
verdict = ua.audit_france(PARQUET)
sysf = ua.system_flags(verdict)
fr_counts = ua.counts(sysf)
print('Systèmes audités :', len(sysf))
print('Stations         :', int(sysf['n_stations'].sum()))
fr_counts

Systèmes audités : 123
Stations         : 46307


{'A1': 17, 'A2': 1, 'A3': 41, 'A4': 88, 'A5': 4, 'A6': 0, 'A7': 32}

## 2. Comparaison avec la colonne FR de la Table 1 du papier

La pipeline librairie doit reproduire **exactement** la colonne FR.

In [3]:
PAPER_FR = {'A1':17,'A2':1,'A3':41,'A4':88,'A5':4,'A6':0,'A7':32}
cmp = pd.DataFrame({'pipeline_librairie': fr_counts, 'papier_Table1_FR': PAPER_FR})
cmp['match'] = cmp['pipeline_librairie'] == cmp['papier_Table1_FR']
cmp

,pipeline_librairie,papier_Table1_FR,match
A1,17,17,True
A2,1,1,True
A3,41,41,True
A4,88,88,True
A5,4,4,True
A6,0,0,True
A7,32,32,True


In [4]:
assert cmp['match'].all(), 'divergence avec la Table 1 FR'
print('OK : la pipeline unique reproduit exactement la colonne FR.')

OK : la pipeline unique reproduit exactement la colonne FR.


## 3. Niveau station + types

In [5]:
st = pd.DataFrame({c: int((verdict[c]==1).sum()) for c in ua.CLASSES},
                  index=['stations_flaggees']).T
types = pd.read_parquet(PARQUET)['station_type'].value_counts()
print('Types de stations:'); print(types)
st

Types de stations:
station_type
free_floating    39235
docked_bike       5442
carsharing        1630
Name: count, dtype: int64


,stations_flaggees
A1,1630
A2,40
A3,39235
A4,500
A5,831
A6,0
A7,32330


## 4. Figure

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,3))
ax.bar(ua.CLASSES, [fr_counts[c] for c in ua.CLASSES], color='#2a9d8f')
ax.set_title('France — systèmes flaggés par classe (pipeline unique)')
ax.set_ylabel('systèmes')
plt.tight_layout()
plt.show()

/tmp/ipykernel_31262/775528012.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Conclusion.** La fonction de la librairie reproduit la colonne FR à l'identique, hors-ligne et de façon déterministe. Le notebook *Global* applique la **même** fonction au monde entier.